# BRAIN-HYBRID — Cerveau Artificiel

**Architecture :** Qwen3-4B (gelé) + 4 modules CfC+SNN + STDP + Hippocampe SDM

Ce notebook :
1. Installe les dépendances
2. Clone le repo et charge le modèle
3. Reprend depuis un checkpoint (si existant)
4. Lance les tests d'intégration GPU
5. Exécute le test d'apprentissage continu (100 steps)
6. Vérifie tous les critères de succès AGENTS.md §8
7. Visualise l'évolution des erreurs

## 1. Setup — Installation + Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch ncps snntorch accelerate tqdm -q

import os, glob, torch

DRIVE_PATH = '/content/drive/MyDrive/brain_hybrid/'
CHECKPOINT_DIR = DRIVE_PATH + 'checkpoints/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Cloner le repo + charger le modèle

In [ ]:
!git clone -b refactor/predictive-coding-20260320-143000-001 https://github.com/Asurelia/SDNC.git /content/sdnc 2>/dev/null || (cd /content/sdnc && git pull)
%cd /content/sdnc

from brain_hybrid.model import BrainHybridModel
from brain_hybrid.config import BrainConfig

config = BrainConfig(model_name='Qwen/Qwen3-4B')
model = BrainHybridModel(config)
print('Modèle chargé')

## 3. Reprise depuis checkpoint (si existant)

In [ ]:
checkpoints = sorted(glob.glob(CHECKPOINT_DIR + 'step_*.pt'))
if checkpoints:
    model.load_state(checkpoints[-1])
else:
    print('Nouveau run — aucun checkpoint trouvé')

## 4. Tests d'intégration GPU

In [ ]:
from brain_hybrid.llm.qwen_wrapper import QwenWrapper

print('--- Test hidden states distincts ---')
reps = model.llm.get_layer_representations('Le chat mange', layers=[8, 36])
sim = torch.nn.functional.cosine_similarity(
    reps[0].float().mean(dim=1), reps[1].float().mean(dim=1)
).item()
print(f'Cosine sim couche 8 vs 36 : {sim:.4f}')
assert sim < 0.95, f'FAIL: sim={sim:.4f} >= 0.95'
print('PASS\n')

print('--- Test forward complet ---')
result = model.forward('Test de fonctionnement', learn=True)
assert 'response' in result
assert 'prediction_errors' in result
assert len(result['prediction_errors']) == 3
print(f'Erreurs : {result["prediction_errors"]}')
print(f'Réponse : {result["response"][:80]}...')
print('PASS\n')

print('--- Test génération ---')
resp = model.llm.generate('Bonjour', max_new_tokens=32)
assert isinstance(resp, str) and len(resp) > 0
print(f'Réponse : {resp[:80]}')
print('PASS\n')

print('--- Test STDP ---')
from brain_hybrid.core.stdp import STDPLearning
stdp = STDPLearning()
w = torch.randn(32, 32)
w_orig = w.clone()
stdp.apply(w, torch.randn(1, 10, 64), torch.randn(1, 10, 32))
assert not torch.allclose(w, w_orig), 'Poids inchangés'
rel_change = (w - w_orig).abs().mean() / w_orig.abs().mean()
assert rel_change < 0.01, f'Changement trop grand: {rel_change:.4f}'
print(f'Changement relatif : {rel_change:.6f}')
print('PASS\n')

print('--- Test hippocampe ---')
from brain_hybrid.memory.hippocampus import HippocampalMemory
h = HippocampalMemory(address_dim=64, content_dim=128, n_locations=1000)
emb = torch.randn(128)
content = torch.randn(128)
h.write(emb, content)
recalled = h.read(emb)
sim_h = torch.nn.functional.cosine_similarity(
    content.unsqueeze(0), recalled.unsqueeze(0)
).item()
assert sim_h > 0.5, f'Recall trop faible: {sim_h:.3f}'
print(f'Cosine sim write/read : {sim_h:.4f}')
print('PASS\n')

vram = torch.cuda.memory_allocated(0) / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM : {vram:.2f} / {vram_total:.1f} GB ({vram/vram_total*100:.0f}%)')
print('\n=== TOUS LES TESTS D\'INTÉGRATION PASSENT ===')

## 5. Test d'apprentissage continu (100 steps)

In [ ]:
from brain_hybrid.eval.continual_test import run_continual_test

results = run_continual_test(model, n_steps=100, verbose=True)

# Sauvegarder le checkpoint
path = f"{CHECKPOINT_DIR}step_{model.step_count:06d}.pt"
model.save_state(path)

## 6. Visualisation

In [ ]:
import matplotlib.pyplot as plt

if len(model.error_history) > 10:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Erreur de prédiction
    axes[0].plot(model.error_history, alpha=0.7)
    axes[0].set_title('Erreur de prédiction moyenne')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Erreur')
    axes[0].grid(True, alpha=0.3)

    # Dopamine
    dop_vals = [s.dopamine_signal for s in model.stdp_learners]
    axes[1].bar([f'Module {i}' for i in range(len(dop_vals))], dop_vals)
    axes[1].set_title('Signal dopamine par module')
    axes[1].set_ylabel('Dopamine')
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f'Erreur initiale  : {model.error_history[0]:.4f}')
    print(f'Erreur finale    : {model.error_history[-1]:.4f}')
    if model.error_history[0] > 0:
        print(f'Amélioration     : {(1 - model.error_history[-1]/model.error_history[0])*100:.1f}%')
    print(f'Mémoires         : {model.hippocampus.stats()}')

## 7. Test mémoire épisodique

In [ ]:
# Apprendre des faits
model.forward('Mon chat s\'appelle Luna et il est roux', learn=True)
model.forward('Luna adore jouer avec des balles de laine', learn=True)

# Interroger avec un fragment
recalled = model.remember('chat Luna')
print(f'Souvenir récupéré — norme : {recalled.norm().item():.3f}')
print(f'Mémoires totales : {len(model.hippocampus.metadata)}')
print(f'Stats hippocampe : {model.hippocampus.stats()}')

## 8. Boucle d'apprentissage libre (optionnel)

In [ ]:
# Décommenter pour lancer une boucle d'apprentissage plus longue

# prompts = [
#     'Explique comment fonctionne la photosynthèse',
#     'Qu\'est-ce que la conscience selon les neurosciences ?',
#     'Comment le cerveau consolide-t-il les souvenirs ?',
#     'Décris le fonctionnement d\'un neurone biologique',
#     'Quelle est la différence entre mémoire courte et longue durée ?',
# ]
# 
# start = model.step_count
# for step in range(start, start + 500):
#     prompt = prompts[step % len(prompts)]
#     result = model.forward(prompt, learn=True)
#     if step % 10 == 0:
#         print(f'Step {step} | err={result["mean_error"]:.4f} | mem={result["memories_stored"]}')
#     if step % 50 == 0 and step > start:
#         model.save_state(f'{CHECKPOINT_DIR}step_{step:06d}.pt')